In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict
from utils import *
import seaborn as sns
from typing import Iterable

In [ ]:

def plot_ceg_stackplot(ax, graph_year_min=2017, graph_year_max=2025, C=None,
                       innovations=None, label_offset=1.7, 
                       annotation_size=12 ,
                       label_size=12 * 1.2,
                       tick_size=10 * 1.4,
                       show_ylabel=True):
    if innovations is None:
        innovations = INNOVATIONS


    if C is None:
        C_min = np.floor(ytc(graph_year_min))
        C_max = np.ceil(ytc(graph_year_max))
        C = np.array(sorted(list(np.logspace(np.log10(C_min), np.log10(C_max), 1000))
                             + [C_min, C_max]))
    C_label = ytc(graph_year_max)
    
    ceg_functions = compute_ceg_functions(C, innovations=innovations)
    cumulative_ceg_functions = compute_ceg_functions(C, innovations=innovations, cumulative=True)
    
    innovation_keys = list(innovations.keys())
    stackable_layers = [cumulative_ceg_functions[innovation_keys[0]]]
    for i in range(1, len(innovation_keys)):
        stackable_layers.append(cumulative_ceg_functions[innovation_keys[i]] - cumulative_ceg_functions[innovation_keys[i-1]])
    
    ax.stackplot(C, 
                 np.vstack(stackable_layers), 
                 colors=[innovation['color'] for _, innovation in innovations.items()])
    
    previous_y_height, label_y_positions = 1, stackable_layers[0][-1] ** (1/2)
    for innovation_key, innovation_config in innovations.items():
        color, label = innovation_config['color'], innovation_config['label']
        
        cumulative_multiplier = cumulative_ceg_functions[innovation_key][-1]
        stack_height = cumulative_multiplier / previous_y_height
        midpoint = previous_y_height * (stack_height) ** (1/2)
        previous_y_height = cumulative_multiplier
        
        ax.plot([C_label, C_label * label_offset], [midpoint, label_y_positions], 
                color=color, linewidth=1, alpha=0.7, clip_on=False)
        ax.text(C_label * label_offset, label_y_positions, f'{label} ({format_sigfigs(ceg_functions[innovation_key][-1] )}x)', 
                color=color, fontsize=annotation_size, va='center', fontweight='bold', clip_on=False)
        
        label_y_positions *= 2.1
    
    total_gains = cumulative_ceg_functions['chinchilla'][-1]
    total_cagr = total_gains  ** (1/(graph_year_max-graph_year_min)) * 100 - 100
    ax.text(C_label * label_offset, 6, f"Total Efficiency Gains:\n{format_sigfigs(total_gains)}x ({format_sigfigs(total_cagr)}% CAGR)", ha='left', va = 'top', fontsize=annotation_size * 1.1, color='black', fontweight='bold', clip_on=False)
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlim(ytc(graph_year_min), C_label)
    ax.set_ylim(1, 1e5)
    ax.set_xlabel('Model Year', fontsize=label_size, fontstyle='italic')
    if show_ylabel:
        ax.set_ylabel('Compute Equivalent Multiplier\n(Relative to LSTM)', fontsize=label_size, fontstyle='italic')
    
    year_ticks = year_ticks = np.arange(np.ceil(graph_year_min), np.floor(graph_year_max))
    compute_values = [ytc(y) for y in year_ticks]
    ax.set_xticks(compute_values)
    ax.set_xticklabels([int(x) for x in year_ticks], fontsize=tick_size)
    ax.tick_params(labelsize=tick_size)
    ax.minorticks_off()
    
    ax2 = ax.twiny()
    ax2.set_xscale('log')
    ax2.set_xlim(ytc(graph_year_min), C_label)
    ax2.set_xlabel('Compute Frontier (FLOPs)', fontsize=label_size, fontstyle='italic', labelpad = 10)
    ax2.tick_params(labelsize=tick_size)
    ax.set_title("Scale Dependent Changes Dominate Efficiency Gains on Increasing Compute Frontier", fontweight='bold', fontsize = label_size * 1.1)
    ax.text(ytc(2017.1), 3e4, "(a)", fontweight='bold')
    
    ax.grid(True, alpha=0.3)
    
    
    return ax2


fig, ax = plt.subplots(figsize=(14,4))
plot_ceg_stackplot(ax, 2016.9, 2025)


plt.tight_layout()
fig.set_dpi(600)
plt.savefig("figures/scale_dependent_stackplot.png", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
def plot_compute_equivalent_growth(ax, year_min, year_max, compute_value, 
                                   label_size=12, tick_size=12, annotation_size=10, 
                                   label_offset=0.35, show_ylabel=True):
    years = np.linspace(year_min, year_max, 1000)
    
    cumulative_ceg = compute_ceg_functions(
        [compute_value] * 1000, 
        years=years, 
        cumulative=True
    )
    
    individual_ceg = compute_ceg_functions(
        [compute_value] * 1000, 
        years=years, 
        cumulative=False
    )
    
    innovation_keys = list(INNOVATIONS.keys())
    stackable_layers = [cumulative_ceg[innovation_keys[0]]]
    for i in range(1, len(innovation_keys)):
        stackable_layers.append(
            cumulative_ceg[innovation_keys[i]] - cumulative_ceg[innovation_keys[i-1]]
        )
    
    ax.stackplot(
        years, 
        np.vstack(stackable_layers), 
        colors=[innovation['color'] for innovation in INNOVATIONS.values()]
    )
    
    previous_y_height = 1
    label_y_positions = stackable_layers[0][-1] ** (1/2)
    year_label = year_max
    
    for innovation_key in innovation_keys:
        color = INNOVATIONS[innovation_key]['color']
        label = INNOVATIONS[innovation_key]['label']
        
        cumulative_multiplier = cumulative_ceg[innovation_key][-1]
        stack_height = cumulative_multiplier / previous_y_height
        midpoint = previous_y_height * (stack_height) ** (1/2)
        previous_y_height = cumulative_multiplier
        
        ax.plot([year_label, year_label + label_offset], [midpoint, label_y_positions], 
                color=color, linewidth=1, alpha=0.7, clip_on=False)
        ax.text(year_label + label_offset, label_y_positions, 
                f'{label} ({format_sigfigs(individual_ceg[innovation_key][-1])}x)', 
                color=color, fontsize=annotation_size, va='center', 
                fontweight='bold', clip_on=False)
        
        label_y_positions *= 2.1
    
    final_value = cumulative_ceg['chinchilla'][-1]
    total_cagr = final_value ** (1/(year_max-year_min)) * 100 - 100
    
    ax.text(year_label + label_offset, 2, 
            f"Total Efficiency Gains:\n{format_sigfigs(final_value)}x ({format_sigfigs(total_cagr)}% CAGR)", 
            ha='left', va='top', fontsize=annotation_size * 1.1, 
            color='black', fontweight='bold', clip_on=False)
    
    ax.set_yscale('log')
    ax.set_ylim(1, 1e5)
    if show_ylabel:
        ax.set_ylabel("Compute Equivalent Multiplier\n(Relative to LSTM)", fontsize=label_size)
    ax.tick_params(labelsize=tick_size)
    ax.grid(True, which='major', alpha=0.3)
    ax.set_xlim(year_min, year_max)
    ax.minorticks_off()
    
    exponent = round(np.log10(compute_value))
    assert(exponent == np.log10(compute_value))
    ax.set_title(rf"Decomposed Efficiency Gains at $10^{{{exponent}}}$ FLOPs, Over Time", 
                 fontsize=label_size * 1.2, fontweight='bold')
    ax.text(2017.1, 3e4, "(b)", fontweight='bold')
    
    return ax

fig, ax = plt.subplots(figsize=(14, 7))
plot_compute_equivalent_growth(ax, 2016.5, 2025, 1e18)

In [ ]:
from matplotlib.colors import Normalize

def plot_compute_silhouette(ax, year_min, year_max, compute_values,
                           ylabel_fontsize=16, tick_fontsize=12, annotation_size=12, 
                           label_offset=.35, show_ylabel=True, show_cagr=True, 
                           linewidth=2.5, labels_inside=False, show_gridlines = True,
                           subplot_label = True):
    granularity = 1000
    years = np.linspace(year_min, year_max, granularity)
    compute_values = list(map(np.float64, compute_values))
    
    cmap = plt.cm.viridis_r
    color_offset = 2
    norm = Normalize(vmin=0, vmax=len(compute_values)-1 + color_offset)
    line_endpoints = []
    
    for i, c in enumerate(compute_values):
        cumulative_ceg = compute_ceg_functions(
            [c] * granularity, 
            years=years, 
            cumulative=True
        )
        silouette = cumulative_ceg['chinchilla']
        color = cmap(norm(i + color_offset))
        
        ax.plot(years, silouette, color=color, linewidth=linewidth)
        
        cagr = ((silouette[-1] / silouette[0]) ** (1 / (year_max - year_min)) - 1) * 100
        line_endpoints.append((silouette[-1], color, int(np.log10(float(c))), cagr))
    
    ax.set_yscale('log')
    ax.set_xlim(year_min, year_max)
    ax.set_ylim(1, 1e5)
    if show_ylabel:
        ax.set_ylabel("Compute Equivalent Multiplier\n(Relative to LSTM)", fontsize=ylabel_fontsize)
    ax.tick_params(labelsize=tick_fontsize)
    if show_gridlines:
        ax.grid(True, which='major', alpha=0.3)
    else:
        ax.grid(False)

    year_range = year_max - year_min
    if year_range <= 10:
        ax.set_xticks(range(int(np.ceil(year_min)), int(year_max) + 1))
    else:
        ax.set_xticks(range(int(np.ceil(year_min)), int(year_max) + 1, 2))
    
    if labels_inside:
        label_x = 2022.5
        label_ha = 'left'
    else:
        label_x = year_max + label_offset
        label_ha = 'left'
    
    for y_val, color, log_c, cagr in line_endpoints:
        if show_cagr:
            label_text = f'$\\mathbf{{10^{{{log_c}}}}}$ FLOPs ({cagr:.0f}% CAGR)'
        else:
            label_text = f'$\\mathbf{{10^{{{log_c}}}}}$ FLOPs'
        ax.text(label_x, y_val * (10 ** 0.15), label_text, 
                color=color, fontsize=annotation_size, 
                va='center', ha=label_ha)

    ax.set_title(rf"Efficiency gains over time with fixed compute".title(), 
                 fontsize=ylabel_fontsize * 1.2, fontweight='bold')
    
    if subplot_label:
        ax.text(2017.1, 3e4, "(c)", fontweight='bold')
    
    return ax

SUBSTACK_PHOTO = True
if SUBSTACK_PHOTO:
    labels_inside = True
    annotation_size = 20
    figsize = (6 ,6 * 3)
    linewidth = 5
    subplot_label = False
    show_cagr = False
    show_gridlines = False
else:
    figsize = (14,4)
    linewidth = 1
    annotation_size = 15
    subplot_label = True
    show_cagr = True
    show_gridlines = True


fig, ax = plt.subplots(figsize=figsize)
compute_values = [np.float64(10**i) for i in range(16, 25, 2)]
fig.set_dpi(300)

plot_compute_silhouette(ax, 2016.5, 2025, compute_values, linewidth = linewidth, labels_inside=True, show_cagr = show_cagr, annotation_size  = annotation_size, subplot_label = subplot_label, show_gridlines= show_gridlines)
plt.tight_layout()

In [ ]:


LABEL_SIZE, TICK_SIZE, ANNOTATION_SIZE = 15, 13, 11
GRAPH_YEAR_MIN, GRAPH_YEAR_MAX = 2016.5, 2025


fig, axs = plt.subplots(nrows = 3, ncols = 1, figsize=(14, 10))

plot_ceg_stackplot(axs[0], GRAPH_YEAR_MIN, GRAPH_YEAR_MAX, label_size=LABEL_SIZE, tick_size = TICK_SIZE, annotation_size=ANNOTATION_SIZE, show_ylabel = False)

compute_level = 1e18
plot_compute_equivalent_growth(axs[1], GRAPH_YEAR_MIN,GRAPH_YEAR_MAX, compute_level, label_size=LABEL_SIZE, tick_size = TICK_SIZE, annotation_size=ANNOTATION_SIZE, show_ylabel=False)

compute_values = [10**i for i in range(16, 25,2)]
plot_compute_silhouette(axs[2], GRAPH_YEAR_MIN,GRAPH_YEAR_MAX, compute_values, ylabel_fontsize = LABEL_SIZE, tick_fontsize=TICK_SIZE, annotation_size=ANNOTATION_SIZE + 2, show_ylabel=False)


fig.supylabel("Compute Equivalent Multiplier\n(Relative to LSTM)", fontsize=LABEL_SIZE * 1.2, x=0.02, ha='center')


plt.tight_layout()
fig.set_dpi(600) 
plt.savefig("figures/combined_scale_dependence_graphs.png", dpi = 600)
plt.show()